# 🧠 Machine Learning Model Training & Comparative Benchmarking
## Predicting Segment Travel Time & Fuel Consumption with AQI
This notebook evaluates Linear Regression, Ridge, Random Forest, and Gradient Boosting models, inspects feature importances, and verifies model serialization.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load preprocessed arrays
npz_path = os.path.join('..', 'data', 'processed', 'train_test_data.npz')
data = np.load(npz_path)

X_train = data['X_train']
X_test = data['X_test']
y_train_time = data['y_train_time']
y_test_time = data['y_test_time']
y_train_fuel = data['y_train_fuel']
y_test_fuel = data['y_test_fuel']

preprocessor = joblib.load(os.path.join('..', 'data', 'processed', 'preprocessor.joblib'))
feature_names = preprocessor.feature_names
print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

### 1. Travel Time Model Benchmark Comparison

In [ ]:
models = {
    "Linear Regression (Baseline)": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=6, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train_time)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test_time, preds)
    rmse = np.sqrt(mean_squared_error(y_test_time, preds))
    r2 = r2_score(y_test_time, preds)
    results.append({"Model": name, "MAE (min)": round(mae, 3), "RMSE": round(rmse, 3), "R² Score": round(r2, 4)})

pd.DataFrame(results)

### 2. Feature Importances for Travel Time Prediction

In [ ]:
gb_model = models["Gradient Boosting"]
importances = gb_model.feature_importances_
sorted_indices = np.argsort(importances)[::-1]

print("Top 10 Most Influential Features:")
for rank, idx in enumerate(sorted_indices[:10], 1):
    print(f"{rank:2d}. {feature_names[idx]:30s} -> {importances[idx]:.4f}")